In [13]:
from dataclasses import dataclass, field
from typing import Optional
import torch
from datasets import load_dataset
from torch.optim import Adam
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    RobertaForSequenceClassification,
    RobertaTokenizer,
    set_seed,
)
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer, create_reference_model
from trl.core import LengthSampler

tqdm.pandas()

In [14]:
@dataclass
class ScriptArguments:
    model_name: Optional[str] = field(default="EleutherAI/pythia-410m-deduped", metadata={"help": "the model name"})
    log_with: Optional[str] = field(default=None, metadata={"help": "use 'wandb' to log with wandb"})
    learning_rate: Optional[float] = field(default=1e-5, metadata={"help": "the learning rate"})
    mini_batch_size: Optional[int] = field(default=4, metadata={"help": "the PPO minibatch size"})
    batch_size: Optional[int] = field(default=16, metadata={"help": "the batch size"})
    gradient_accumulation_steps: Optional[int] = field(default=1, metadata={"help": "the number of gradient accumulation steps"})
    model_save_path: Optional[str] = field(default="./pythia-detoxified-final", metadata={"help": "the path to save the model"})

# Ignore the environment arguments

In [15]:
import sys
sys.argv = sys.argv[:1]

from transformers import HfArgumentParser
parser = HfArgumentParser(ScriptArguments)
script_args = parser.parse_args_into_dataclasses()[0]

# Configuration PPO

In [18]:
config = PPOConfig(
    learning_rate=script_args.learning_rate,
    mini_batch_size=script_args.mini_batch_size,
    batch_size=script_args.batch_size,
    gradient_accumulation_steps=script_args.gradient_accumulation_steps,
    output_dir=script_args.model_save_path,
)

# Function to build dataset for training

In [21]:
def build_dataset(config, dataset_name="civil_comments", input_min_text_length=5, input_max_text_length=10):
    tokenizer = AutoTokenizer.from_pretrained(script_args.model_name)
    tokenizer.pad_token = tokenizer.eos_token

    # Charger le dataset
    ds = load_dataset(dataset_name)

    # Sélectionner le split 'train' et effectuer le prétraitement
    ds_train = ds['train']

    # Filtrer les exemples en fonction de la longueur du texte
    ds_train = ds_train.filter(lambda x: len(x["text"]) > input_min_text_length and len(x["text"]) < input_max_text_length)
    
    input_size = LengthSampler(input_min_text_length, input_max_text_length)

    # Tokenisation
    def tokenize(sample):
        input_text = sample["text"]
        sample["input_ids"] = tokenizer.encode(input_text)[: input_size()]
        sample["query"] = tokenizer.decode(sample["input_ids"])
        return sample

    ds_train = ds_train.map(tokenize, batched=False)
    ds_train.set_format(type="torch")

    # Séparer en train et test
    ds_train = ds_train.train_test_split(test_size=0.2, shuffle=False)["train"]
    
    return ds_train

# Build dataset

In [22]:
min_input_length = 30
max_input_length = 40
dataset = build_dataset(config, input_min_text_length=min_input_length, input_max_text_length=max_input_length)
set_seed(config.seed)

Filter:   0%|          | 0/1804874 [00:00<?, ? examples/s]

Map:   0%|          | 0/46080 [00:00<?, ? examples/s]

# Load the model

In [23]:
model = AutoModelForCausalLM.from_pretrained(script_args.model_name, torch_dtype=torch.bfloat16)
model = AutoModelForCausalLMWithValueHead.from_pretrained(model)

# Create the reference model

In [24]:
ref_model = create_reference_model(model, num_shared_layers=20)

# Adam optimizer

In [25]:
optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=config.learning_rate)

# Load the tokenizer

In [26]:
tokenizer = AutoTokenizer.from_pretrained(script_args.model_name)
tokenizer.pad_token = tokenizer.eos_token

# PPOTrainer

In [ ]:
ppo_trainer = PPOTrainer(
    config=config,
    model=model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    reward_model=None,
    train_dataset=dataset,
    data_collator=lambda data: {key: [d[key] for d in data] for key in data[0]},
)

# Toxicity model for reward calculation

In [35]:
toxicity_model_id = "facebook/roberta-hate-speech-dynabench-r4-target"
toxicity_tokenizer = RobertaTokenizer.from_pretrained(toxicity_model_id)
toxicity_model = RobertaForSequenceClassification.from_pretrained(toxicity_model_id, torch_dtype=torch.float16).to(
    ppo_trainer.accelerator.device
)

generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}

output_min_length = 20
output_max_length = 30
output_length_sampler = LengthSampler(output_min_length, output_max_length)

tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/816 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

NameError: name 'ppo_trainer' is not defined

# Training loop

In [ ]:
for epoch, batch in tqdm(enumerate(ppo_trainer.dataloader)):
    query_tensors = batch["input_ids"]

    response_tensors = []
    for query in query_tensors:
        gen_len = output_length_sampler()
        generation_kwargs["max_new_tokens"] = gen_len
        response = ppo_trainer.generate(query, **generation_kwargs)
        response_tensors.append(response.squeeze()[-gen_len:])
    
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    # Get toxicity scores
    texts = batch["response"]
    toxicity_inputs = toxicity_tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(
        ppo_trainer.accelerator.device
    )
    logits = toxicity_model(**toxicity_inputs).logits.float()
    toxicity_labels = (logits[:, 0]).tolist()

    rewards = [torch.tensor(output) for output in toxicity_labels]

    # Run PPO step
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
    ppo_trainer.log_stats(stats, batch, rewards)

    # Save model every 100 epochs
    if epoch % 100 == 0:
        if ppo_trainer.accelerator.is_main_process:
            ppo_trainer.save_pretrained(script_args.model_save_path)

# Save the model after training completes

In [ ]:
ppo_trainer.save_pretrained(script_args.model_save_path)

---

In [3]:
!pip install transformers trl datasets detoxify accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 7.6 MB/s eta 0:00:00:00:01
